In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Dimension Users

In [0]:
user_df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "s3://pipeline-project-uk/bronze/location/user") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
        .option("cloudFiles.schemaHints",        
            "user_id INT, user_name STRING, country STRING, subscription_type STRING, start_date DATE, end_date DATE, updated_at TIMESTAMP")\
    .load("s3://pipeline-project-uk/bronze/DimUser")

In [0]:
user_df = user_df \
    .withColumn("user_name", initcap(trim(col("user_name")))) \
    .withColumn("country", trim(col("country"))) \
    .withColumn("subscription_type", upper(trim(col("subscription_type")))) \
    .withColumn("start_date", to_date(col("start_date"))) \
    .withColumn("end_date", to_date(col("end_date"))) \
    .withColumn("updated_at", to_timestamp(col("updated_at")))

In [0]:
user_df = user_df\
    .withColumn("subscription_start_year", year(col("start_date"))) \
    .withColumn("subscription_start_month", month(col("start_date"))) \
    .withColumn("days_since_last_update", datediff(current_date(), col("updated_at")))

In [0]:
user_df = user_df.select(
    "user_id",
    "user_name",
    "country",
    "subscription_type",
    "start_date",
    "end_date",
    "subscription_start_year",
    "subscription_start_month",
    "days_since_last_update",
    "updated_at"              
)

In [0]:
user_df.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", "s3://pipeline-project-uk/bronze/Destination/DimUser/checkpoint")\
        .option("path", "s3://pipeline-project-uk/bronze/Destination/DimUser/data")\
        .trigger(availableNow=True)\
        .start()

### Dimension Artist

In [0]:
artistSchema= "artist_id INT, artist_name STRING, genre STRING, country STRING, updated_at TIMESTAMP"

In [0]:
artist_df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "s3://pipeline-project-uk/bronze/location/artist") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
        .option("cloudFiles.schemaHints",artistSchema)\
    .load("s3://pipeline-project-uk/bronze/DimArtist")

In [0]:
artist_df = artist_df.withColumn("artist_name", initcap(trim(col("artist_name")))) \
    .withColumn("genre",       initcap(trim(col("genre")))) \
    .withColumn("country",     trim(col("country"))) \
    .withColumn("updated_at",  to_timestamp(col("updated_at")))

In [0]:
artist_df = artist_df \
    .withColumn("first_name", split(col("artist_name"), " ")[0]) \
    .withColumn("last_name",  element_at(split(col("artist_name"), " "), -1))

In [0]:
artist_df = artist_df \
    .withColumn("genre_category",
                when(col("genre").isin("Rock", "Pop", "Electronic"),lit("Modern"))
               .when(col("genre").isin("Jazz", "Classical"),lit("Traditional"))
               .when(col("genre") == "Hip-Hop",lit("Urban"))
               .otherwise(lit("Other")))

In [0]:
artist_df = artist_df.withColumn("last_updated_year", year(col("updated_at"))) \
    .withColumn("last_updated_month", month(col("updated_at"))) \
    .withColumn("days_since_last_update", datediff(current_date(), col("updated_at")))

In [0]:
artist_df = artist_df.select(
    "artist_id",
    "artist_name",
    "genre",
    "country",
    "first_name",
    "last_name",
    "genre_category",
    "last_updated_year",
    "last_updated_month",
    "days_since_last_update",
    "updated_at"                  
)

In [0]:
artist_df.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", "s3://pipeline-project-uk/bronze/Destination/DimArtist/checkpoint")\
        .option("path", "s3://pipeline-project-uk/bronze/Destination/DimArtist/data")\
        .trigger(availableNow=True)\
        .start()

### Dimenstion Track

In [0]:
trackSchema="track_id INT, track_name STRING, artist_id INT, album_name STRING, duration_sec INT, release_date DATE, updated_at TIMESTAMP"

In [0]:
track_df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "s3://pipeline-project-uk/bronze/location/track") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
        .option("cloudFiles.schemaHints",trackSchema)\
    .load("s3://pipeline-project-uk/bronze/DimTrack")

In [0]:
track_df = track_df.withColumn("track_name",initcap(trim(col("track_name"))))\
    .withColumn("album_name",   initcap(trim(col("album_name"))))\
    .withColumn("release_date", to_date(col("release_date")))\
    .withColumn("updated_at",   to_timestamp(col("updated_at")))

In [0]:
track_df=track_df.withColumn("duration_min", round(col("duration_sec") / 60, 2))\
    .withColumn("duration_formatted",                                          
                concat_ws(":", 
                    lpad((col("duration_sec") / 60).cast("int").cast("string"), 2, "0"),
                    lpad((col("duration_sec") % 60).cast("string"), 2, "0")))\
    .withColumn("track_length_category",
                when(col("duration_sec") < 120,  lit("Short"))       # < 2 mins
               .when(col("duration_sec") < 240,  lit("Standard"))    # 2–4 mins
               .when(col("duration_sec") < 360,  lit("Long"))        # 4–6 mins
               .otherwise(lit("Extended")))

In [0]:
track_df=track_df.withColumn("release_year",  year(col("release_date")))\
    .withColumn("release_month", month(col("release_date")))\
    .withColumn("release_quarter", quarter(col("release_date")))

In [0]:
track_df=track_df.withColumn("days_since_last_update", datediff(current_date(), col("updated_at")))

In [0]:
track_df = track_df.select(
    "track_id",
    "track_name",
    "artist_id",
    "album_name",
    "duration_sec",
    "duration_min",
    "duration_formatted",
    "track_length_category",
    "release_date",
    "release_year",
    "release_month",
    "release_quarter",
    "days_since_last_update",
    "updated_at"             
)

In [0]:
track_df.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", "s3://pipeline-project-uk/bronze/Destination/DimTrack/checkpoint")\
        .option("path", "s3://pipeline-project-uk/bronze/Destination/DimTrack/data")\
        .trigger(availableNow=True)\
        .start()

### Dimension Silver

In [0]:
dateSchema = "date_key INT, date DATE, day INT, month INT, year INT, weekday STRING"

In [0]:
date_df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "s3://pipeline-project-uk/bronze/location/date") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
        .option("cloudFiles.schemaHints",dateSchema)\
    .load("s3://pipeline-project-uk/bronze/DimDate")

In [0]:
date_df = date_df.withColumn("month_name",date_format(col("date"), "MMMM")) \
           .withColumn("is_weekend", col("weekday").isin("Saturday", "Sunday")) \
           .withColumn("start_of_quarter",to_date(date_trunc("quarter", col("date"))))


In [0]:
date_df.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", "s3://pipeline-project-uk/bronze/Destination/DimDate/checkpoint")\
        .option("path", "s3://pipeline-project-uk/bronze/Destination/DimDate/data")\
        .trigger(availableNow=True)\
        .start()

### Fact Table

In [0]:
factSchema="stream_id LONG, user_id INT, track_id INT, date_key INT, listen_duration INT, device_type STRING, stream_timestamp TIMESTAMP"

In [0]:
fact_df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "s3://pipeline-project-uk/bronze/location/fact") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
        .option("cloudFiles.schemaHints",factSchema)\
    .load("s3://pipeline-project-uk/bronze/FactStream")

In [0]:
fact_df = fact_df.withColumn("is_carryable", when(col("device_type") == "Desktop", "No")\
    .when((col("device_type") == "Mobile") | (col("device_type") == "Smart Speaker"), "Yes")\
    .otherwise("No Data Found"))



In [0]:
    # Arranging the Columns
    fact_df = fact_df.select(
        "stream_id",
        "user_id",
        "track_id",
        "date_key",
        "listen_duration",
        "device_type",
        "is_carryable",       
        "stream_timestamp"    
        )

In [0]:
fact_df.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", "s3://pipeline-project-uk/bronze/Destination/FactData/checkpoint")\
        .option("path", "s3://pipeline-project-uk/bronze/Destination/FactData/data")\
        .trigger(availableNow=True)\
        .start()